# 🧠 Modeling Phase — Multiclass Classification

### **Overview**
We train and evaluate machine learning models to predict **failure types** for machines.  
The dataset is imbalanced, so SMOTE is used to balance the classes.

**Steps Covered:**
1. Prepare Data & Encode Target
2. Handle Class Imbalance (SMOTE)
3. Train Random Forest
4. Train XGBoost
5. Evaluate Models


## 📌 Loading the Preprocessed Dataset

After completing all preprocessing steps (handling missing values, outlier removal, feature scaling, and PCA) in the data preprocessing notebook, we saved the final cleaned dataset as:

``preprocessed_smart_data.csv``

In this modeling notebook, we simply load the processed file instead of repeating the entire preprocessing pipeline:




In [1]:
import pandas as pd

hd_clean = pd.read_csv("preprocessed_smart_data.csv")
print("Loaded processed dataset:", hd_clean.shape)


Loaded processed dataset: (98538, 32)


## Prepare Data & Encode Target

In [2]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Target column
y_class = hd_clean['failure_type']

# Encode target for multiclass classification
le = LabelEncoder()
y_encoded = le.fit_transform(y_class)

# Features
X = hd_clean[['temperature', 'vibration', 'humidity', 'pressure', 'energy_consumption',
              'machine_status', 'anomaly_flag', 'downtime_risk']]

# Encode categorical features
X = pd.get_dummies(X.join(hd_clean[['machine_id']]), drop_first=True)

# Feature scaling
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train/Test split
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)


## 2️⃣ Handle Class Imbalance
- The dataset is imbalanced across failure types.
- SMOTE (Synthetic Minority Oversampling Technique) is used to balance the training data.


In [48]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)


## 3️⃣ Train Random Forest
- Use `RandomForestClassifier` with 300 trees.  
- `class_weight='balanced'` is used to handle class imbalance.


In [49]:
from sklearn.ensemble import RandomForestClassifier

rf_clf = RandomForestClassifier(n_estimators=300, random_state=42, class_weight='balanced')
rf_clf.fit(X_train_res, y_train_res)


RandomForestClassifier(class_weight='balanced', n_estimators=300,
                       random_state=42)

## Random Forest — Base Model Evaluation

- Evaluate the Random Forest model on the test set.
- Metrics: Accuracy, F1-score (macro & weighted), Confusion Matrix, Classification Report.


In [50]:
# Evaluate Random Forest
y_pred_rf = rf_clf.predict(X_test)

from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

acc_rf = accuracy_score(y_test, y_pred_rf)
f1_macro_rf = f1_score(y_test, y_pred_rf, average='macro')
f1_weighted_rf = f1_score(y_test, y_pred_rf, average='weighted')
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("===== Random Forest Evaluation =====")
print(f"Accuracy: {acc_rf}")
print(f"F1 Macro: {f1_macro_rf}")
print(f"F1 Weighted: {f1_weighted_rf}")
print("\nConfusion Matrix:\n", cm_rf)
print("\nClassification Report:\n", classification_report(y_test, y_pred_rf))


===== Random Forest Evaluation =====
Accuracy: 0.9235335904201339
F1 Macro: 0.38350040418339787
F1 Weighted: 0.9302579212563626

Confusion Matrix:
 [[   28     0    56    38    78]
 [   44 17727    78    81   182]
 [   43     0   108    93   148]
 [   42     0    94    79   173]
 [   53     1   150   153   259]]

Classification Report:
               precision    recall  f1-score   support

           0       0.13      0.14      0.14       200
           1       1.00      0.98      0.99     18112
           2       0.22      0.28      0.25       392
           3       0.18      0.20      0.19       388
           4       0.31      0.42      0.36       616

    accuracy                           0.92     19708
   macro avg       0.37      0.40      0.38     19708
weighted avg       0.94      0.92      0.93     19708



## 4️⃣ Train XGBoost
- Use `XGBClassifier` with 300 trees and `mlogloss` as eval metric.
- Disable `use_label_encoder` for recent versions of XGBoost.


In [51]:
from xgboost import XGBClassifier

xgb_clf = XGBClassifier(n_estimators=300, random_state=42, use_label_encoder=False, eval_metric='mlogloss')
xgb_clf.fit(X_train_res, y_train_res)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='mlogloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=None, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=None, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=300, n_jobs=None,
              num_parallel_tree=None, ...)

## XGBoost — Base Model Evaluation

- Evaluate the XGBoost model on the test set.


In [52]:
# Evaluate XGBoost
y_pred_xgb = xgb_clf.predict(X_test)

acc_xgb = accuracy_score(y_test, y_pred_xgb)
f1_macro_xgb = f1_score(y_test, y_pred_xgb, average='macro')
f1_weighted_xgb = f1_score(y_test, y_pred_xgb, average='weighted')
cm_xgb = confusion_matrix(y_test, y_pred_xgb)

print("===== XGBoost Evaluation =====")
print(f"Accuracy: {acc_xgb}")
print(f"F1 Macro: {f1_macro_xgb}")
print(f"F1 Weighted: {f1_weighted_xgb}")
print("\nConfusion Matrix:\n", cm_xgb)
print("\nClassification Report:\n", classification_report(y_test, y_pred_xgb))


===== XGBoost Evaluation =====
Accuracy: 0.9224172924700629
F1 Macro: 0.37843906855958265
F1 Weighted: 0.9290468783836362

Confusion Matrix:
 [[   31     5    60    43    61]
 [   54 17738    79    91   150]
 [   59    11   108    93   121]
 [   47     6   100    87   148]
 [   81     8   157   155   215]]

Classification Report:
               precision    recall  f1-score   support

           0       0.11      0.15      0.13       200
           1       1.00      0.98      0.99     18112
           2       0.21      0.28      0.24       392
           3       0.19      0.22      0.20       388
           4       0.31      0.35      0.33       616

    accuracy                           0.92     19708
   macro avg       0.36      0.40      0.38     19708
weighted avg       0.94      0.92      0.93     19708



## 5️⃣ Train LightGBM

- Use **LGBMClassifier** for multiclass classification.  
- Parameters:
  - `n_estimators=300`  
  - `learning_rate=0.1`  
  - `max_depth=7`  
  - `random_state=42`  
- Training is done on **resampled training data** (after SMOTE).


In [53]:
from lightgbm import LGBMClassifier

# Initialize LightGBM classifier
lgbm_clf = LGBMClassifier(
    n_estimators=300,
    learning_rate=0.1,
    max_depth=7,
    random_state=42
)

# Train on SMOTE-resampled data
lgbm_clf.fit(X_train_res, y_train_res)


[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035999 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1540
[LightGBM] [Info] Number of data points in the train set: 362230, number of used features: 9
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Info] Start training from score -1.609438
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf


LGBMClassifier(max_depth=7, n_estimators=300, random_state=42)

## LightGBM — Base Model Evaluation

- Evaluate the LightGBM model on the test set.


In [54]:
# Evaluate LightGBM
y_pred_lgbm = lgbm_clf.predict(X_test)

acc_lgbm = accuracy_score(y_test, y_pred_lgbm)
f1_macro_lgbm = f1_score(y_test, y_pred_lgbm, average='macro')
f1_weighted_lgbm = f1_score(y_test, y_pred_lgbm, average='weighted')
cm_lgbm = confusion_matrix(y_test, y_pred_lgbm)

print("===== LightGBM Evaluation =====")
print(f"Accuracy: {acc_lgbm}")
print(f"F1 Macro: {f1_macro_lgbm}")
print(f"F1 Weighted: {f1_weighted_lgbm}")
print("\nConfusion Matrix:\n", cm_lgbm)
print("\nClassification Report:\n", classification_report(y_test, y_pred_lgbm))


===== LightGBM Evaluation =====
Accuracy: 0.9213517353359042
F1 Macro: 0.3731999053448959
F1 Weighted: 0.9270034745681688

Confusion Matrix:
 [[   42    14    57    38    49]
 [   65 17755    90    84   118]
 [   71    23   100    86   112]
 [   62    16    94    87   129]
 [   99    29   170   144   174]]

Classification Report:
               precision    recall  f1-score   support

           0       0.12      0.21      0.16       200
           1       1.00      0.98      0.99     18112
           2       0.20      0.26      0.22       392
           3       0.20      0.22      0.21       388
           4       0.30      0.28      0.29       616

    accuracy                           0.92     19708
   macro avg       0.36      0.39      0.37     19708
weighted avg       0.93      0.92      0.93     19708



# 🔍 Hyperparameter Tuning & Best Model Selection (Simplified)

### 1️⃣ Overview
- Tune hyperparameters for **Random Forest, XGBoost, and LightGBM** using **`RandomizedSearchCV`** with `n_iter=10`.
- Faster and lighter version suitable for Colab.
- Evaluate all tuned models on the **test set**.
- Compare metrics (F1-macro) and select the **best-performing model**.
- Save the best model as a **Pickle file** (`best_model_machine_fail.pkl`).

---

### 2️⃣ Step 1: Randomized Hyperparameter Tuning
- Use **RandomizedSearchCV** instead of GridSearchCV.
- Limit the number of iterations and parameter ranges to reduce runtime.
- Train each model on the **SMOTE-resampled training data**.

---

### 3️⃣ Step 2: Evaluate Tuned Models
- Evaluate each tuned model on the test set.
- Metrics: **Accuracy, F1-score (macro & weighted), Confusion Matrix, Classification Report**.
- This helps to identify the best-performing model.

---

### 4️⃣ Step 3: Select Best Model
- Compare the F1-macro of all three models.
- Select the model with the highest F1-macro as the **best model**.

---

### 5️⃣ Step 4: Save Best Model
- Save the selected best model as a **Pickle file** for future use.
- Filename: `best_model_machine_fail.pkl`


# 🔹 Random Forest — Randomized Hyperparameter Tuning

- Tune Random Forest hyperparameters using `RandomizedSearchCV`.
- n_iter=10 for faster execution.
- Evaluate the best Random Forest model later.


In [ ]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

# Parameter grid for Random Forest
rf_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 15],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "max_features": ["sqrt", "log2"],
    "class_weight": ["balanced"]
}

# Randomized Search
rf_rand = RandomizedSearchCV(
    RandomForestClassifier(random_state=42),
    rf_params,
    n_iter=10,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=42
)
rf_rand.fit(X_train_res, y_train_res)

# Train best model
best_rf = RandomForestClassifier(**rf_rand.best_params_, random_state=42)
best_rf.fit(X_train_res, y_train_res)

print("✅ Random Forest tuning completed. Best params:", rf_rand.best_params_)


# 🔹 XGBoost — Randomized Hyperparameter Tuning

- Tune XGBoost hyperparameters using `RandomizedSearchCV`.
- n_iter=10 for faster execution.


In [ ]:
from xgboost import XGBClassifier

xgb_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [4, 6],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.8, 1.0],
    "colsample_bytree": [0.8, 1.0]
}

xgb_rand = RandomizedSearchCV(
    XGBClassifier(objective="multi:softprob", eval_metric="mlogloss", random_state=42),
    xgb_params,
    n_iter=10,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=42
)
xgb_rand.fit(X_train_res, y_train_res)

best_xgb = XGBClassifier(
    **xgb_rand.best_params_,
    objective="multi:softprob",
    eval_metric="mlogloss",
    random_state=42
)
best_xgb.fit(X_train_res, y_train_res)

print("✅ XGBoost tuning completed. Best params:", xgb_rand.best_params_)


# 🔹 LightGBM — Randomized Hyperparameter Tuning

- Tune LightGBM hyperparameters using `RandomizedSearchCV`.
- n_iter=10 for faster execution.


In [ ]:
from lightgbm import LGBMClassifier

lgbm_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [5, 7],
    "learning_rate": [0.05, 0.1],
    "num_leaves": [20, 31]
}

lgbm_rand = RandomizedSearchCV(
    LGBMClassifier(random_state=42),
    lgbm_params,
    n_iter=10,
    cv=3,
    scoring="f1_macro",
    n_jobs=-1,
    random_state=42
)
lgbm_rand.fit(X_train_res, y_train_res)

best_lgbm = LGBMClassifier(**lgbm_rand.best_params_, random_state=42)
best_lgbm.fit(X_train_res, y_train_res)

print("✅ LightGBM tuning completed. Best params:", lgbm_rand.best_params_)


# 🔹 Evaluate Tuned Models & Select Best Model

- Evaluate **Random Forest, XGBoost, and LightGBM** after hyperparameter tuning.
- Metrics: Accuracy, F1-score (macro & weighted), Confusion Matrix, Classification Report.
- Select the **best-performing model** based on **F1-macro**.


In [ ]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Function to evaluate a model
def evaluate_model(model, X_test, y_test, name="Model"):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    f1_weighted = f1_score(y_test, y_pred, average="weighted")
    cm = confusion_matrix(y_test, y_pred)

    print(f"===== {name} Report =====")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Macro: {f1_macro:.4f}")
    print(f"F1 Weighted: {f1_weighted:.4f}")
    print("\nConfusion Matrix:\n", cm)
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("\n")

    return f1_macro

# Evaluate each tuned model individually
rf_f1 = evaluate_model(best_rf, X_test, y_test, "Random Forest")
xgb_f1 = evaluate_model(best_xgb, X_test, y_test, "XGBoost")
lgbm_f1 = evaluate_model(best_lgbm, X_test, y_test, "LightGBM")

# Compare and select the best model based on F1-macro
model_list = [(best_rf, "Random Forest"), (best_xgb, "XGBoost"), (best_lgbm, "LightGBM")]
best_model, best_model_name = max(model_list, key=lambda x: f1_score(y_test, x[0].predict(X_test), average="macro"))

print(f"🔥 Best Model Selected: {best_model_name}")


# 🔹 Compare Models & Select Best

- Evaluate **Random Forest, XGBoost, and LightGBM** on the test set.
- Compare **F1-macro** to select the best-performing model.


In [56]:
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix

# Function to evaluate a model
def evaluate_model(model, X_test, y_test, name="Model"):
    y_pred = model.predict(X_test)
    acc = accuracy_score(y_test, y_pred)
    f1_macro = f1_score(y_test, y_pred, average="macro")
    f1_weighted = f1_score(y_test, y_pred, average="weighted")
    cm = confusion_matrix(y_test, y_pred)

    print(f"===== {name} Evaluation =====")
    print(f"Accuracy: {acc:.4f}")
    print(f"F1 Macro: {f1_macro:.4f}")
    print(f"F1 Weighted: {f1_weighted:.4f}")
    print("\nConfusion Matrix:\n", cm)
    print("\nClassification Report:\n", classification_report(y_test, y_pred))
    print("\n")

    return f1_macro

# Evaluate all models
f1_rf = evaluate_model(rf_clf, X_test, y_test, "Random Forest")
f1_xgb = evaluate_model(xgb_clf, X_test, y_test, "XGBoost")
f1_lgbm = evaluate_model(lgbm_clf, X_test, y_test, "LightGBM")

# Compare F1-macro and select best model
model_list = [(rf_clf, "Random Forest", f1_rf),
              (xgb_clf, "XGBoost", f1_xgb),
              (lgbm_clf, "LightGBM", f1_lgbm)]

best_model, best_model_name, best_f1 = max(model_list, key=lambda x: x[2])

print(f"🔥 Best Model Selected: {best_model_name} with F1-macro: {best_f1:.4f}")


===== Random Forest Evaluation =====
Accuracy: 0.9235
F1 Macro: 0.3835
F1 Weighted: 0.9303

Confusion Matrix:
 [[   28     0    56    38    78]
 [   44 17727    78    81   182]
 [   43     0   108    93   148]
 [   42     0    94    79   173]
 [   53     1   150   153   259]]

Classification Report:
               precision    recall  f1-score   support

           0       0.13      0.14      0.14       200
           1       1.00      0.98      0.99     18112
           2       0.22      0.28      0.25       392
           3       0.18      0.20      0.19       388
           4       0.31      0.42      0.36       616

    accuracy                           0.92     19708
   macro avg       0.37      0.40      0.38     19708
weighted avg       0.94      0.92      0.93     19708



===== XGBoost Evaluation =====
Accuracy: 0.9224
F1 Macro: 0.3784
F1 Weighted: 0.9290

Confusion Matrix:
 [[   31     5    60    43    61]
 [   54 17738    79    91   150]
 [   59    11   108    93   121]
 [ 

## Save Best Model

- Save the best-performing model as a Pickle file for deployment.
- Example filename: `best_model_failure_type.pkl`


In [57]:
import pickle

# Save the best model
with open("best_model_failure_type.pkl", "wb") as f:
    pickle.dump(best_model, f)

print(f"✅ {best_model_name} saved as best_model_failure_type.pkl")


✅ Random Forest saved as best_model_failure_type.pkl


# 🛠️ Final Insights for Milestone 3: Predictive Maintenance Model Development and Optimization

## 🎯 Objectives
- Build, train, and optimize the predictive maintenance model for forecasting equipment failures.

## 📝 Tasks

### 1️⃣ Model Selection
- Choose a suitable model for time-series forecasting based on the nature of the data:
  - **LSTM (Long Short-Term Memory networks)** 🧠: Good for sequential data and time-series predictions.
  - **Random Forest** 🌲: Robust to overfitting and can handle non-linear relationships in the data.
  - **XGBoost** ⚡: A powerful gradient boosting model for classification and regression tasks.
  - **LightGBM** 🌟: Efficient gradient boosting framework, fast training, handles large datasets and categorical features well.

### 2️⃣ Model Training
- Split the data into **training, validation, and test sets** to evaluate model performance effectively.
- Use **time-series cross-validation** ⏱️ to ensure the model generalizes well across different time periods and avoids overfitting.

### 3️⃣ Model Evaluation
- Evaluate the models using metrics such as:
  - **Precision** 🎯: Measures the accuracy of positive predictions (i.e., correctly predicting equipment failure).
  - **Recall** 🔍: Measures the ability to capture all true failures.
  - **F1-score** ⚖️: The harmonic mean of precision and recall, balancing the two.
  - **ROC-AUC** 📊: Measures the model’s ability to distinguish between failure and non-failure instances.

### 4️⃣ Model Optimization
- Fine-tune **hyperparameters** 🔧 using **Grid Search** or **Random Search** to improve model performance.
- Explore combining multiple models via **ensemble methods** 🤝 to boost prediction accuracy.

## 💡 Key Takeaways
- LSTM is ideal for sequential trends in sensor data.  
- Random Forest is robust for mixed-feature datasets with non-linear patterns.  
- XGBoost and LightGBM excel in boosting performance and handling large datasets.  
- Hyperparameter tuning and ensemble methods significantly improve prediction accuracy.  
- Time-series cross-validation ensures models generalize well across operational periods.  
